### RAG Pipelines - Data Ingestion to Vector DB

In [30]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [31]:
#read all pdfs

def process_all_pdfs(pdf_dir):
    """Process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_dir)

    # find all pdf file recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            #add metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'
            
            all_documents.extend(documents)
            print(f"loaded {len(documents)} pages")
        
        except Exception as e:
            print(f"Error: {e}")

    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: 2024-wttc-introduction-to-ai.pdf
loaded 44 pages

Processing: 271_AI Lect Notes.pdf
loaded 128 pages

Total documents loaded: 172


In [32]:
all_pdf_documents

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTR

### Chunk

In [33]:
## text split into chunks

#chunk size - controls how much text is placed into each chunk.
# chunk overlap - controls how much text is repeated between consecutive chunks to preserve context.(eg:200 means, last 200 words starts with beginning of next chunk)

def split_doc(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function = len, 
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

      # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [34]:
chunks=split_doc(all_pdf_documents)
chunks


Split 172 documents into 498 chunks

Example chunk:
Content: INTRODUCTION TO AI
World Travel & Tourism Council
< Contents  | 1
INTRODUCTION 
TO ARTIFICIAL 
INTELLIGENCE (AI) 
TECHNOLOGY
GUIDE FOR TRAVEL & TOURISM LEADERS
January 2024...
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTR

### Embedding and Vector Store

In [35]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""
    def __init__(self, model_name: str = "all-miniLm-L6-v2"):
        """ 
        Initialize the embedding manager
         
         Args:
            model_name: HuggingFace model name for sentence embeddings
            
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray :
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
                raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-miniLm-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3054.07it/s]


Model loaded successfully. Embedding dimension: 384


### Vector store

In [37]:
class vectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self,collection_name: str = "pdf_documents",persist_directory: str = "../data/vector_store"):
        """
                Initialize the vector store
                
                Args:
                    collection_name: Name of the ChromaDB collection
                    persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #unique id generate 
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
        
        #add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise  

vectorstore = vectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 498


In [38]:
chunks

[Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 0, 'page_label': '1', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 1\nINTRODUCTION \nTO ARTIFICIAL \nINTELLIGENCE (AI) \nTECHNOLOGY\nGUIDE FOR TRAVEL & TOURISM LEADERS\nJanuary 2024'),
 Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': '..\\data\\pdf\\2024-wttc-introduction-to-ai.pdf', 'total_pages': 44, 'page': 1, 'page_label': '2', 'source_file': '2024-wttc-introduction-to-ai.pdf', 'file_type': 'pdf'}, page_content='INTR

In [39]:

### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 498 texts...


Batches: 100%|██████████| 16/16 [00:35<00:00,  2.22s/it]


Generated embeddings with shape: (498, 384)
Adding 498 documents to vector store...
Successfully added 498 documents to vector store
Total documents in collection: 996


### Retriever Pipeline From VectorStore (query->embedding->retrive from vector)

In [40]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: vectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold : float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        #generate query embeddings
        query_embedding = self.embedding_manager.generate_embeddings(query)

        #search in vector db
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            #process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                    
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [41]:

rag_retriever

In [43]:

rag_retriever.retrieve("But where has AI come from and why is it suddenly such big news?")

Retrieving documents for query: 'But where has AI come from and why is it suddenly such big news?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 64 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.80it/s]

Generated embeddings with shape: (384,)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)


[{'id': 'doc_369a3108_6',
  'content': 'INTRODUCTION TO AI\nWorld Travel & Tourism Council\n4\n< Contents  |\n“…It happened gradually, then suddenly…”\nThis famous quote from Ernest Hemingway’s 1926 novel, The Sun Also Rises could be used to describe some of \nthe world’s most profound technological changes. Small advancements accumulate and then all of a sudden, \nthe world is a different place. \nThat could describe the world before and after the birth of the internet and could now be applied to Artificial \nIntelligence (AI), which has ‘gradually, then suddenly’ burst onto the scene after nearly a century of research. It \nis almost impossible to browse the news or social media today, without seeing mention of Artificial Intelligence \nand the magazines Time, Science, Cosmopolitan and The Economist (to name only a few) have all dedicated \ncover stories to AI in 2023.\nBut where has AI come from and why is it suddenly such big news?',
  'metadata': {'page_label': '4',
   'creationda

### Integration Vectordb Context pipeline With LLM output (context -> llm -> output)

In [46]:
#simple rag pipeline with GROQ llm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#initialize llm 
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024 )


## 2. Simple RAG function: retrieve context + generate response

def rag_simple(query, retriever, llm, top_k=3):
    #retrive the context
    results = retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content



In [47]:
answer = rag_simple("But where has AI come from and why is it suddenly such big news?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'But where has AI come from and why is it suddenly such big news?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 64 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.24it/s]


Generated embeddings with shape: (384,)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
AI has come from nearly a century of research and is suddenly big news because small advancements have accumulated, making it burst onto the scene, with sophisticated AI chatbots and accessibility through desktop computers, smartphones, and the cloud, exposing its power to the general public.


### Enhance RAG pipeline

In [51]:
def rag_advanced(query, retriever, llm , top_k = 5, min_score = 0.2, return_context= True):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    #prepare context & source
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

  # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

result = rag_advanced("While ChatGPT was the first AI chatbot to reach mass consumption and what are others availabe now ?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])



Retrieving documents for query: 'While ChatGPT was the first AI chatbot to reach mass consumption and what are others availabe now ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 99 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.33it/s]

Generated embeddings with shape: (384,)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


Answer: While ChatGPT was the first AI chatbot to reach mass consumption, others now available include Google Bard, IBM Watsonx.ai, Microsoft Copilot, and Ernie AI.
Sources: [{'source': '2024-wttc-introduction-to-ai.pdf', 'page': 6, 'score': 0.38063013553619385, 'preview': 'INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 7\nArgentinian footballer Lionel Messi : Deepfake Messi Messages\nAI has therefore been regularly hitting the news headlines and as with all new trends, a few high profile cases \ncan propel a technology that is initially only used by a fe...'}, {'source': '2024-wttc-introduction-to-ai.pdf', 'page': 6, 'score': 0.38063013553619385, 'preview': 'INTRODUCTION TO AI\nWorld Travel & Tourism Council\n< Contents  | 7\nArgentinian footballer Lionel Messi : Deepfake Messi Messages\nAI has therefore been regularly hitting the news headlines and as with all new trends, a few high profile cases \ncan propel a technology that is initially only used by a fe...'}, {

# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

In [53]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What is AI, How it works?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What is AI, How it works?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 25 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.74it/s]

Generated embeddings with shape: (384,)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
One of the 

most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”. This means AI 
systems appear to think, learn and act like humans and in some cases exceed the capabilities of humans. AI 
systems can analyse vast amounts of data, solve complex problems, make decisions and perform creative tasks.
FAKE FAKE

One of the most important aspects of AI is that it is a multi-use technology. Like electricity it can be applied 
in lots of different ways, to lots of different scenarios.
There is no single, universally accepted definition for Artificial Intelligence, but the Oxford English Dictionary 
defines AI as “the capacity of computers, or other machines, to exhibit intelligent behaviour”